In [32]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model 
from langchain.agents import create_agent 
load_dotenv()  # Load environment variables from .env file

os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

model = init_chat_model("gpt-4.1") # initialize a chat model with the name "gpt-4"
model 

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.7.3', 'langchain': '1.4.7', 'langchain-openai': '1.7.3'}}, profile={'name': 'GPT-4.1', 'release_date': '2025-04-14', 'last_updated': '2025-04-14', 'open_weights': False, 'max_input_tokens': 1047576, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True, 'mid_conversation_system_messages': True, 'mid_conversation_tools': False}, client=<openai.resources.chat.completions.completions.Completions object at 0x11101d310>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x11101d6d0>

# Pydantic 

Pydantic is used for structured outputs very similar to zod library used in typeScript. 

In [10]:
from pydantic import BaseModel, Field

class Movie(BaseModel): #inheriting from BaseModel to create a structured output model for the movie information
    title: str = Field(description="The title of the movie")
    year: int = Field(description="The year the movie was released")
    director: str = Field(description="The director of the movie")
    genre: str = Field(description="The genre of the movie")
    rating: float = Field(description="The rating of the movie on a scale of 1 to 10 (assumed)")

In [13]:
# we are telling the model that we want the output to be structured according to the Movie class. 
# This means that the model will return a Movie object instead of a plain text response.
model_with_structured_output = model.with_structured_output(Movie) 
response = model_with_structured_output.invoke("Tell me about the upcoming movie Ramayana the legend pf prince rama.") # invoke the model with a prompt to get information about the movie Inception
response


Movie(title='Ramayana: The Legend of Prince Rama', year=2024, director='Kunal Kohli (Live-action adaptation in development); Original animated film by Ram Mohan, Koichi Sasaki, Yugo Sako', genre='Mythology, Animation (original), Action, Adventure', rating=7.5)

## Message output alongside Parsed output

In [18]:
class MovieDetails(BaseModel): #inheriting from BaseModel to create a structured output model for the movie information
    title: str = Field(description="The title of the movie")
    year: int = Field(description="The year the movie was released")
    director: str = Field(description="The director of the movie")
    genre: str = Field(description="The genre of the movie")
    rating: float = Field(description="The rating of the movie on a scale of 1 to 10 (assumed)")
    reason_for_rating: str = Field(description="The reason for the rating given to the movie")

#include_raw=True means that the model will return the raw text output from the model in addition to the structured output.
#  This can be useful for debugging or for getting additional information from the model that may not fit into the structured output format.
model_output = model.with_structured_output(MovieDetails, include_raw=True) 

response = model_output.invoke("Tell me about the bollywood movie Aadipurush") # invoke the model with a prompt to get information about the movie Inception
response

{'raw': AIMessage(content='{"title":"Adipurush","year":2023,"director":"Om Raut","genre":"Mythological, Action, Drama","rating":4.0,"reason_for_rating":"The film received mixed to negative reviews due to its dialogues, visual effects, and adaptation choices, but it was noted for its ambitious scale and performances by some of the lead actors. It is a retelling of the Ramayana with modern VFX, leading to polarized reactions."}', additional_kwargs={'parsed': MovieDetails(title='Adipurush', year=2023, director='Om Raut', genre='Mythological, Action, Drama', rating=4.0, reason_for_rating='The film received mixed to negative reviews due to its dialogues, visual effects, and adaptation choices, but it was noted for its ambitious scale and performances by some of the lead actors. It is a retelling of the Ramayana with modern VFX, leading to polarized reactions.'), 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 94, 'prompt_tokens': 181, 'total_tokens': 275, 'completi

## Nested Structure


In [20]:
class Actor(BaseModel):
    name: str = Field(description="The name of the actor")
    age: int = Field(description="The age of the actor")
    role: str = Field(description="The role of the actor in the movie")

class MovieWithActors(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="The year the movie was released")
    director: str = Field(description="The director of the movie")
    genre: str = Field(description="The genre of the movie")
    budget: str | None = Field(default=None, description="The budget of the movie in dollars(optional)") # We used a default value of None for the budget field to make it optional.
    actors: list[Actor] = Field(description="A list of actors in the movie") # We used the actor class as a type for the actors field.

model_output_with_actors = model.with_structured_output(MovieWithActors, include_raw=True)
response = model_output_with_actors.invoke("Tell me about the movie interstellar and its cast.") 
response 

{'raw': AIMessage(content='{"title":"Interstellar","year":2014,"director":"Christopher Nolan","genre":"Science Fiction, Adventure, Drama","budget":"$165,000,000","actors":[{"name":"Matthew McConaughey","age":54,"role":"Joseph Cooper"},{"name":"Anne Hathaway","age":41,"role":"Dr. Amelia Brand"},{"name":"Jessica Chastain","age":47,"role":"Murph"},{"name":"Michael Caine","age":91,"role":"Professor Brand"},{"name":"Matt Damon","age":53,"role":"Dr. Mann"},{"name":"Mackenzie Foy","age":23,"role":"Young Murph"}]}', additional_kwargs={'parsed': MovieWithActors(title='Interstellar', year=2014, director='Christopher Nolan', genre='Science Fiction, Adventure, Drama', budget='$165,000,000', actors=[Actor(name='Matthew McConaughey', age=54, role='Joseph Cooper'), Actor(name='Anne Hathaway', age=41, role='Dr. Amelia Brand'), Actor(name='Jessica Chastain', age=47, role='Murph'), Actor(name='Michael Caine', age=91, role='Professor Brand'), Actor(name='Matt Damon', age=53, role='Dr. Mann'), Actor(name=

## TypedDict
it is a way to define a dictionary with specific key-value pairs and their types. It is useful for creating structured data that can be validated and type-checked.

### Different between TypedDict and Pydantic:
The main difference between TypedDict and Pydantic is that TypedDict is a way to define a dictionary with specific key-value pairs and their types, 
while Pydantic is a library for data validation and settings management using Python type annotations.

**🌟 TypeDict provides a simpler alternative using python's built-in typing, ideal when u don't need runtime validation.**

- dict       → actual unstructured data
- TypedDict  → typed description of a dictionary (describes an expected shape of the data, with type hints)
- Pydantic   → validated structured object (It automatically validates expected data or convert data at runtime.)


In [25]:
from typing_extensions import TypedDict, Annotated #annoted is used to add metadata to the fields of the TypedDict, which can be used for validation or documentation purposes.

class MovieWithActorsTypedDict(TypedDict):
    title: Annotated[str, ..., "The title of the movie"] # ... is used to indicate that the optional fields are empty.
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    genre: Annotated[str, ..., "The genre of the movie"]
    budget: Annotated[str | None, ..., "The budget of the movie in dollars(optional)"] 

model_with_typedDict = model.with_structured_output(MovieWithActorsTypedDict)
response = model_with_typedDict.invoke("Tell me about the movie Avengers Endgame.")
response


{'title': 'Avengers: Endgame',
 'year': 2019,
 'director': 'Anthony Russo, Joe Russo',
 'genre': 'Superhero, Action, Sci-Fi',
 'budget': '$356 million'}

In [27]:
class Actor(TypedDict):
    name: str 
    age: int
    role: str 

class MovieWithActors(TypedDict):
    title: str 
    year: int
    director: str 
    genre: str = "Unknown"
    budget: str | None = None
    actors: list[Actor] = []

model_output_with_actors = model.with_structured_output(MovieWithActors)
response = model_output_with_actors.invoke("Tell me about the movie interstellar and its cast.") 
response 

{'title': 'Interstellar',
 'year': 2014,
 'director': 'Christopher Nolan',
 'genre': 'Science Fiction, Adventure, Drama',
 'budget': 'Approx. $165 million',
 'actors': [{'name': 'Matthew McConaughey', 'age': 44, 'role': 'Cooper'},
  {'name': 'Anne Hathaway', 'age': 31, 'role': 'Amelia Brand'},
  {'name': 'Jessica Chastain', 'age': 37, 'role': 'Murph (adult)'},
  {'name': 'Michael Caine', 'age': 81, 'role': 'Professor Brand'},
  {'name': 'Mackenzie Foy', 'age': 14, 'role': 'Young Murph'},
  {'name': 'Casey Affleck', 'age': 39, 'role': 'Tom (adult)'},
  {'name': 'Matt Damon', 'age': 44, 'role': 'Dr. Mann'},
  {'name': 'John Lithgow', 'age': 69, 'role': 'Donald'}]}

In [ ]:
model.profile #show the model is performing in terms of latency, throughput, and other metrics. This can be useful for monitoring the performance of the model and identifying any potential issues.

{'name': 'GPT-4.1',
 'release_date': '2025-04-14',
 'last_updated': '2025-04-14',
 'open_weights': False,
 'max_input_tokens': 1047576,
 'max_output_tokens': 32768,
 'text_inputs': True,
 'image_inputs': True,
 'audio_inputs': False,
 'pdf_inputs': True,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': False,
 'tool_calling': True,
 'structured_output': True,
 'attachment': True,
 'temperature': True,
 'image_url_inputs': True,
 'pdf_tool_message': True,
 'image_tool_message': True,
 'tool_choice': True,
 'tool_call_streaming': True,
 'mid_conversation_system_messages': True,
 'mid_conversation_tools': False}

## How to use Pydantic and TypedDict with create_agent

In [ ]:
from pydantic import BaseModel, Field

class ContactInfo(BaseModel):
    """Contact information of a person"""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model= "gpt-4.1",
    response_format=ContactInfo # this is the way we attach a structured output with create_agent. The response_format parameter is used to specify the structured output format for the agent's responses. In this case, we are using the ContactInfo class as the structured output format.
)

result = agent.invoke({
    "messages": [
        {"role": "user", "content": "What is the contact information for John Doe?"}
    ]
})

print(result["structured_response"])  # This will print the structured output in the form of a ContactInfo object.


name='John Doe' email='johndoe@email.com' phone='555-123-4567'


In [ ]:
from typing_extensions import TypedDict

# no type validation, just defined the schema for the structured output
# This is useful when you want to define a structured output format without enforcing type validation. 
# It allows you to specify the expected structure of the output without requiring the model to strictly adhere to it.
class ContactInfo(TypedDict):
    """Contact information of a person"""
    name: str
    email: str
    phone: str
    phone: str

agent = create_agent(
    model= "gpt-4.1",
    response_format=ContactInfo # this is the way we attach a structured output with create_agent. The response_format parameter is used to specify the structured output format for the agent's responses. In this case, we are using the ContactInfo class as the structured output format.
)

result = agent.invoke({
    "messages": [
        {"role": "user", "content": "What is the contact information for John Doe?"}
    ]
})

print(result["structured_response"])  # This will print the structured output in the form of a ContactInfo object.


{'name': 'John Doe', 'email': 'johndoe@email.com', 'phone': '555-123-4567'}


## DataClasses

A data class is a class typically containing mainly data, there  aren't really an restrictions. You create it using @dataclass decorator.

In [40]:
from dataclasses import dataclass
from langchain.agents import create_agent


@dataclass
class ContactInfo:
    """Contact information of a person"""
    name: str
    email: str
    phone: str

agent = create_agent(
    model= "gpt-4.1",
    response_format=ContactInfo # this is the way we attach a structured output with create_agent. The response_format parameter is used to specify the structured output format for the agent's responses. In this case, we are using the ContactInfo class as the structured output format.
)
result = agent.invoke({
    "messages": [
        {"role": "user", "content": "What is the contact information for John Doe?"}
    ]
})  
print(result["structured_response"])  # This will print the structured output in the form of a ContactInfo object.

ContactInfo(name='John Doe', email='john.doe@example.com', phone='555-123-4567')
